In [1]:
from ultralytics import YOLO
import cv2
import numpy as np

# Load YOLOv8 Model
model = YOLO("yolov8n.pt")   

# Input and Output Video
video_path = "traffic.mp4"          
output_path = "vehicle_counting_output.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Counting Line
line_y = height // 2

# Store counted IDs
counted_ids = set()
total_count = 0

# COCO Classes
vehicle_classes = {
    2: "Car",
    7: "Truck"
}

# Process Video
while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    # Draw counting line
    cv2.line(frame, (0, line_y), (width, line_y), (0,255,255), 3)

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls in zip(boxes, ids, classes):
            if cls not in vehicle_classes:
                continue

            x1, y1, x2, y2 = map(int, box)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            label = vehicle_classes[cls]

            # Draw Box
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

            cv2.putText(
                frame,
                f"{label} ID:{track_id}",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

            cv2.circle(frame, (cx,cy), 4, (0,0,255), -1)

            # Count vehicle only once
            if abs(cy - line_y) < 5:
                if track_id not in counted_ids:
                    counted_ids.add(track_id)
                    total_count += 1

    # Display Count
    cv2.putText(
        frame,
        f"Total Vehicles: {total_count}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,0,0),
        3
    )

    writer.write(frame)

    cv2.imshow("Vehicle Counting", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
writer.release()
cv2.destroyAllWindows()

print("Processing Complete!")
print("Total Vehicles Counted:", total_count)
print("Saved Video:", output_path)

Processing Complete!
Total Vehicles Counted: 16
Saved Video: vehicle_counting_output.mp4
